# How this paper was made

This notebook makes the part of the paper that describes the paper itself:
the repository is treated as the data, and every item below is measured
from it rather than written by hand.

| Paper item | What it shows |
|---|---|
| Table 2 | Provenance: the notebook, inputs and output behind every generated item |
| Figure 2 | Where each file in the repository is stored (plain git or Git LFS), by size |
| Numbers in Section 4 | Counts of generated items, pinned packages and LFS files |

Everything measured here lies **outside `paper/`**. If the measurements
included `paper/`, pasting the new numbers into `main.tex` would change
them again, and the paper could never be up to date.

In [ ]:
import json
import re
import subprocess
import sys
from pathlib import Path

# Make the repository's shared helpers importable from this notebook
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from common import paths
from common.text import count_macros

paths.use_paper_style()
ROOT = paths.REPO_ROOT

## Read what every notebook produces

Each analysis notebook's code is scanned for the calls that make paper
outputs: `save_paper_figure(fig, "<name>")` for figures,
`show_for_paper(tex, "<location>")` for tables and numbers, and
`paths.DATA_DIR / "<file>"` for the data it reads.

In [ ]:
FIGURE_CALL = re.compile(r'save_paper_figure\(\s*\w+\s*,\s*"([^"]+)"')
PASTE_CALL = re.compile(r'show_for_paper\(\s*\w+\s*,\s*"([^"]+)"')
DATA_READ = re.compile(r'DATA_DIR\s*/\s*"([^"]+)"')


def paper_item(output):
    """Name the paper item from a figure file name or a paste location."""
    if m := re.match(r"figure_(\d+)", output):
        return f"Figure {m.group(1)}"
    if m := re.search(r"Table (\d+)", output):
        return f"Table {m.group(1)}"
    return "Numbers in the text"


rows = []
notebooks = sorted((ROOT / "analysis").glob("*.ipynb"))
for nb_path in notebooks:
    source = "\n".join("".join(c["source"]) for c in json.loads(nb_path.read_text())["cells"]
                       if c["cell_type"] == "code")
    inputs = sorted(set(DATA_READ.findall(source)))
    for name in FIGURE_CALL.findall(source):
        rows.append((paper_item(name), nb_path.name, inputs, f"{name}.pdf"))
    for location in PASTE_CALL.findall(source):
        rows.append((paper_item(location), nb_path.name, inputs, "pasted into main.tex"))

rows.sort(key=lambda r: (r[0].split()[0] != "Figure", r[0].split()[0] != "Table", r[0]))
for row in rows:
    print(row)

## Table 2: provenance of every generated item

In [ ]:
def tt(text):
    """Typewriter text with LaTeX-safe underscores."""
    return r"\texttt{" + text.replace("_", r"\_") + "}"

lines = [
    r"\begin{tabular}{lll}",
    r"\hline",
    r"Item & Produced by & Reads \\",
    r"\hline",
]
for item, notebook, inputs, _ in rows:
    reads = ", ".join(tt(f) for f in inputs) if inputs else "the repository"
    lines.append(f"{item} & {tt(notebook)} & {reads} \\\\")
lines += [r"\hline", r"\end{tabular}"]
table_tex = "\n".join(lines) + "\n"

paths.show_for_paper(table_tex, "the table* environment for Table 2")

## Figure 2: where each file is stored

Every file tracked by git outside `paper/`, by size and role, coloured by
whether it is stored directly in git or with Git LFS. For an LFS file the
size is read from its pointer, so this works whether or not the LFS content
has been downloaded.

In [ ]:
def git(*args):
    return subprocess.run(["git", *args], cwd=ROOT, capture_output=True,
                          text=True, check=True).stdout


def stored_in_lfs(path):
    return git("check-attr", "filter", "--", path).strip().endswith("lfs")


def file_size(path):
    """Size in bytes; for an LFS pointer, the size of the real file."""
    text = (ROOT / path).read_bytes()[:200]
    if text.startswith(b"version https://git-lfs"):
        return int(re.search(rb"size (\d+)", text).group(1))
    return (ROOT / path).stat().st_size


def role(path):
    if path.startswith("data/") and not path.endswith((".md", "LICENSE")):
        return "Data products"
    if path.startswith(("analysis/", "common/")):
        return "Analysis code"
    if path.endswith(".md") or path.endswith("LICENSE"):
        return "Documentation"
    return "Configuration"


files = [p for p in git("ls-files").splitlines() if not p.startswith("paper/")]
sizes = np.array([file_size(p) for p in files])
in_lfs = np.array([stored_in_lfs(p) for p in files])
roles = ["Configuration", "Documentation", "Analysis code", "Data products"]
row = np.array([roles.index(role(p)) for p in files])

fig, ax = plt.subplots(figsize=(3.25, 1.9))
rng = np.random.default_rng(0)   # small vertical jitter so points do not overlap
jitter = rng.uniform(-0.15, 0.15, len(files))
for flag, label, marker in [(False, "Plain git", "o"), (True, "Git LFS", "D")]:
    sel = in_lfs == flag
    ax.scatter(sizes[sel], row[sel] + jitter[sel], s=10, marker=marker,
               label=label, zorder=3)

for limit, text in [(1e6, "1 MB"), (50e6, "50 MB"), (100e6, "100 MB")]:
    ax.axvline(limit, color="0.5", ls="--", lw=0.5)
    ax.text(limit * 0.8, -0.4, text, rotation=90, ha="right", va="bottom",
            fontsize=7, color="0.4")

ax.set_xscale("log")
ax.set_xlim(30, 2e8)
ax.set_yticks(range(len(roles)), roles)
ax.set_ylim(-0.5, len(roles) - 0.5)
ax.minorticks_off()
ax.tick_params(axis="y", which="both", left=False, right=False)
ax.set_xlabel("File size [bytes]")
# Legend in the empty space between the small files and the size limits
ax.legend(loc="lower center", bbox_to_anchor=(0.42, 0.0), fontsize=7)

out = paths.save_paper_figure(fig, "figure_2_repository_storage")
print(f"Saved {out.relative_to(paths.REPO_ROOT)}")

## Numbers quoted in Section 4

In [ ]:
env_text = (ROOT / "environment.yml").read_text()
pinned = re.findall(r"^\s*-\s*([A-Za-z0-9_.-]+)=", env_text, flags=re.M)
lfs_sizes = sizes[in_lfs]

# Counts read as words up to nine ("three"), with a capitalised form
# (\NAnalysisNotebooksCap) for the start of a sentence
macros = {
    **count_macros("NAnalysisNotebooks", len(notebooks)),
    **count_macros("NGeneratedFigures", sum(r[0].startswith("Figure") for r in rows)),
    **count_macros("NGeneratedTables", sum(r[0].startswith("Table") for r in rows)),
    **count_macros("NPinnedPackages", len(pinned)),
    **count_macros("NLFSFiles", in_lfs.sum()),
    "LFSDataSize": f"{lfs_sizes.sum() / 1e6:.1f}\\,MB",
}
macro_tex = "% Generated by analysis/how_this_paper_was_made.ipynb -- do not edit by hand\n"
macro_tex += "".join(f"\\newcommand{{\\{k}}}{{{v}}}\n" for k, v in macros.items())

paths.show_for_paper(macro_tex, "the preamble")